In [ ]:
!pip install ucimlrepo

In [ ]:
import pandas as pd
import numpy as np
import joblib, warnings
warnings.filterwarnings('ignore')

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from sklearn.feature_selection import RFE
from sklearn.calibration import CalibratedClassifierCV
from imblearn.combine import SMOTETomek
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
# ── 1. Download All 4 UCI Heart Disease Datasets ─────────────
# UCI repo id=45 includes Cleveland, Hungarian, Switzerland, VA Long Beach
print("Downloading UCI Heart Disease dataset (all sources)...")
heart = fetch_ucirepo(id=45)

X_raw = heart.data.features
y_raw = heart.data.targets
print(f"Total rows fetched: {len(X_raw)}")
print(f"Columns: {list(X_raw.columns)}")



Total rows fetched: 303
Columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


In [ ]:
# ── 2. Select & Rename to Full Column Names ──────────────────
cols_map = {
    'age'      : 'Age',
    'sex'      : 'Sex',
    'cp'       : 'ChestPainType',
    'trestbps' : 'RestingBloodPressure',
    'chol'     : 'Cholesterol',
    'fbs'      : 'FastingBloodSugar',
    'restecg'  : 'RestingECG',
    'thalach'  : 'MaxHeartRate',
    'exang'    : 'ExerciseInducedAngina',
    'oldpeak'  : 'Oldpeak',
    'slope'    : 'SlopeOfST_Segment',
    'ca'       : 'MajorVesselsColored',
    'thal'     : 'Thalassemia'
}

df = X_raw[list(cols_map.keys())].rename(columns=cols_map).copy()
df['HeartDisease'] = (y_raw.values.ravel() > 0).astype(int)

print(f"\nColumns: {list(df.columns)}")
print(f"Shape before cleaning: {df.shape}")


Columns: ['Age', 'Sex', 'ChestPainType', 'RestingBloodPressure', 'Cholesterol', 'FastingBloodSugar', 'RestingECG', 'MaxHeartRate', 'ExerciseInducedAngina', 'Oldpeak', 'SlopeOfST_Segment', 'MajorVesselsColored', 'Thalassemia', 'HeartDisease']
Shape before cleaning: (303, 14)


In [ ]:
# ── 3. Handle Missing Values ─────────────────────────────────
for col in ['MajorVesselsColored', 'Thalassemia', 'SlopeOfST_Segment']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in df.columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

df = df[(df['RestingBloodPressure'] > 0) & (df['Cholesterol'] > 0)]
df = df.drop_duplicates()
print(f"Shape after cleaning: {df.shape}")
print(f"Class distribution:\n{df['HeartDisease'].value_counts()}")


Shape after cleaning: (303, 14)
Class distribution:
HeartDisease
0    164
1    139
Name: count, dtype: int64


In [ ]:
# ── 4. Encoding maps ───────────────────────────────────
encoding_maps = {
    'Sex'                  : {'M': 1,  'F': 0},
    'FastingBloodSugar'    : {'Yes': 1, 'No': 0},
    'ExerciseInducedAngina': {'Yes': 1, 'No': 0},
    'ChestPainType'        : {
        'Typical Angina'    : 0,
        'Atypical Angina'   : 1,
        'Non-Anginal Pain'  : 2,
        'Asymptomatic'      : 3
    },
    'RestingECG'           : {
        'Normal'            : 0,
        'ST-T Abnormality'  : 1,
        'Left Ventricular Hypertrophy': 2
    },
    'SlopeOfST_Segment'    : {
        'Upsloping'         : 1,
        'Flat'              : 2,
        'Downsloping'       : 3
    },
    'Thalassemia'          : {
        'Normal'            : 1,
        'Fixed Defect'      : 2,
        'Reversible Defect' : 3
    }
}
joblib.dump(encoding_maps, 'encoding_maps.pkl')
print("Saved encoding_maps.pkl")


Saved encoding_maps.pkl


In [ ]:
# ── 5. Feature Engineering ───────────────────────────────────
df['Age_MaxHR_Ratio']  = df['Age'] / (df['MaxHeartRate'] + 1)
df['BP_Chol_Ratio']    = df['RestingBloodPressure'] / (df['Cholesterol'] + 1)
df['Age_Oldpeak']      = df['Age'] * df['Oldpeak']
df['MaxHR_Oldpeak']    = df['MaxHeartRate'] / (df['Oldpeak'] + 1)
df['ChestPain_Angina'] = df['ChestPainType'] * df['ExerciseInducedAngina']
df['Slope_Oldpeak']    = df['SlopeOfST_Segment'] * df['Oldpeak']
df['Age_Bin']          = pd.cut(df['Age'], bins=[0,40,50,60,70,100], labels=[0,1,2,3,4]).astype(int)
df['MaxHR_Bin']        = pd.cut(df['MaxHeartRate'], bins=[0,100,130,160,220], labels=[0,1,2,3]).astype(int)

X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

In [ ]:
# ── 5. SMOTETomek ────────────────────────────────────────────
X_res, y_res = SMOTETomek(random_state=42).fit_resample(X, y)
print(f"\nAfter SMOTETomek: {dict(pd.Series(y_res).value_counts())}")


After SMOTETomek: {1: np.int64(143), 0: np.int64(143)}


In [ ]:
# ── 6. Scale & Split ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [ ]:
# ── 7. RFE Feature Selection ────────────────────────────────
print("\nRunning RFE...")
rfe = RFE(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
          n_features_to_select=14, step=1)
rfe.fit(X_train_sc, y_train)
selected = X.columns[rfe.support_].tolist()
print(f"Selected features: {selected}")

X_train_rfe = rfe.transform(X_train_sc)
X_test_rfe  = rfe.transform(X_test_sc)


Running RFE...
Selected features: ['Age', 'ChestPainType', 'RestingBloodPressure', 'Cholesterol', 'MaxHeartRate', 'Oldpeak', 'MajorVesselsColored', 'Thalassemia', 'Age_MaxHR_Ratio', 'BP_Chol_Ratio', 'Age_Oldpeak', 'MaxHR_Oldpeak', 'ChestPain_Angina', 'Slope_Oldpeak']


In [ ]:
# ── 8. Train Models ──────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    gamma=0.2, reg_alpha=0.5, reg_lambda=2.0,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1
)

lgbm = LGBMClassifier(
    n_estimators=500, learning_rate=0.03, num_leaves=20,
    max_depth=5, min_child_samples=30, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
    random_state=42, verbose=-1, n_jobs=-1
)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=7, min_samples_split=10,
    min_samples_leaf=5, random_state=42, n_jobs=-1
)


In [ ]:
# ── 9. Calibrated Stacking ───────────────────────────────────
stack = StackingClassifier(
    estimators=[('xgb', xgb), ('lgbm', lgbm), ('rf', rf)],
    final_estimator=LogisticRegression(C=1.0, max_iter=1000),
    cv=StratifiedKFold(5), n_jobs=-1
)
calibrated = CalibratedClassifierCV(stack, cv=3, method='isotonic')
calibrated.fit(X_train_rfe, y_train)


CalibratedClassifierCV(cv=3,
                       estimator=StackingClassifier(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                                                    estimators=[('xgb',
                                                                 XGBClassifier(base_score=None,
                                                                               booster=None,
                                                                               callbacks=None,
                                                                               colsample_bylevel=None,
                                                                               colsample_bynode=None,
                                                                               colsample_bytree=0.8,
                                                                               device=None,
                                                                               early_stopping_rounds=None,
                                                                               enable_categorical=False,
                                                                               eval_metric='logloss',
                                                                               f...
                                                                                max_depth=5,
                                                                                min_child_samples=30,
                                                                                n_estimators=500,
                                                                                n_jobs=-1,
                                                                                num_leaves=20,
                                                                                random_state=42,
                                                                                reg_alpha=0.5,
                                                                                reg_lambda=2.0,
                                                                                subsample=0.8,
                                                                                verbose=-1)),
                                                                ('rf',
                                                                 RandomForestClassifier(max_depth=7,
                                                                                        min_samples_leaf=5,
                                                                                        min_samples_split=10,
                                                                                        n_estimators=300,
                                                                                        n_jobs=-1,
                                                                                        random_state=42))],
                                                    final_estimator=LogisticRegression(max_iter=1000),
                                                    n_jobs=-1),
                       method='isotonic')

In [ ]:
# ── 10. Optimal Threshold ────────────────────────────────────
y_prob_tr = calibrated.predict_proba(X_train_rfe)[:, 1]
best_thresh, best_acc = 0.5, 0
for t in np.arange(0.3, 0.7, 0.01):
    acc = accuracy_score(y_train, (y_prob_tr >= t).astype(int))
    if acc > best_acc:
        best_acc, best_thresh = acc, t


In [ ]:
# ── 11. Evaluate ─────────────────────────────────────────────
y_prob_test = calibrated.predict_proba(X_test_rfe)[:, 1]
y_pred_opt  = (y_prob_test >= best_thresh).astype(int)

print(f"\n{'='*55}")
print(f"Stacking Ensemble — Optimal Threshold ({best_thresh:.2f})")
print(f"{'='*55}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_opt):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_test):.4f}")
print(classification_report(y_test, y_pred_opt, target_names=['No Disease', 'Heart Disease']))

cv = cross_val_score(calibrated, X_train_rfe, y_train,
                     cv=StratifiedKFold(5), scoring='accuracy', n_jobs=-1)
print(f"5-Fold CV : {cv.mean():.4f} ± {cv.std():.4f}")


Stacking Ensemble — Optimal Threshold (0.32)
Accuracy : 0.8621
ROC-AUC  : 0.9453
               precision    recall  f1-score   support

   No Disease       0.86      0.86      0.86        29
Heart Disease       0.86      0.86      0.86        29

     accuracy                           0.86        58
    macro avg       0.86      0.86      0.86        58
 weighted avg       0.86      0.86      0.86        58

5-Fold CV : 0.8776 ± 0.0462


In [ ]:
# ── 13. Save All Artifacts ───────────────────────────────────
joblib.dump(calibrated,      'best_model.pkl')
joblib.dump(scaler,          'scaler.pkl')
joblib.dump(rfe,             'rfe_selector.pkl')
joblib.dump(best_thresh,     'threshold.pkl')
joblib.dump(list(X.columns), 'feature_names.pkl')
# encoding_maps already saved above
print("\n✅ Saved: best_model.pkl | scaler.pkl | rfe_selector.pkl | threshold.pkl | encoding_maps.pkl | feature_names.pkl")


✅ Saved: best_model.pkl | scaler.pkl | rfe_selector.pkl | threshold.pkl | encoding_maps.pkl | feature_names.pkl


In [ ]:
new = pd.DataFrame([{
    'Age'                  : 55,
    'Sex'                  : 1,
    'ChestPainType'        : 0,
    'RestingBloodPressure' : 140,
    'Cholesterol'          : 230,
    'FastingBloodSugar'    : 1,
    'RestingECG'           : 0,
    'MaxHeartRate'         : 145,
    'ExerciseInducedAngina': 0,
    'Oldpeak'              : 1.5,
    'SlopeOfST_Segment'    : 2,
    'MajorVesselsColored'  : 0,
    'Thalassemia'          : 2
}])
new['Age_MaxHR_Ratio']  = new['Age'] / (new['MaxHeartRate'] + 1)
new['BP_Chol_Ratio']    = new['RestingBloodPressure'] / (new['Cholesterol'] + 1)
new['Age_Oldpeak']      = new['Age'] * new['Oldpeak']
new['MaxHR_Oldpeak']    = new['MaxHeartRate'] / (new['Oldpeak'] + 1)
new['ChestPain_Angina'] = new['ChestPainType'] * new['ExerciseInducedAngina']
new['Slope_Oldpeak']    = new['SlopeOfST_Segment'] * new['Oldpeak']
new['Age_Bin']          = 2
new['MaxHR_Bin']        = 2

new_sc  = scaler.transform(new)
new_rfe = rfe.transform(new_sc)

pred = calibrated.predict(new_rfe)[0]
prob = calibrated.predict_proba(new_rfe)[0][1]
print(f"\nPrediction  : {'Heart Disease' if pred == 1 else 'No Disease'}")
print(f"Probability : {prob*100:.1f}%")


Prediction  : No Disease
Probability : 20.4%
